In [9]:
import os
import json
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

from options import OptionSurface, Deribit, OKX, Bybit
from portfolio_management import Portfolio
from api_client import TradingDeskAPI
from scanner import MarketScanner

In [10]:
# Initialize all classes and parameters

load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
JWT_TOKEN = os.getenv("JWT")
DATA_DIR = os.getenv("DATA_DIR")
FRACTION = float(os.getenv("FRACTION"))
MAX_POSITION = float(os.getenv("MAX_POSITION"))
ENTRY_EV_THRESHOLD = float(os.getenv("ENTRY_EV_THRESHOLD")) # require 1% edge, default = 0
EXIT_EV_THRESHOLD = float(os.getenv("EXIT_EV_THRESHOLD"))
print(BASE_URL)

with open(f"{DATA_DIR}/crypto_tag_ids.json", "r") as f:
    crypto_tag_ids = set(json.load(f))

s = OptionSurface()
portfolio = Portfolio()
api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD, token=JWT_TOKEN)
scanner = MarketScanner(api=api)

target_expiry_str = "25DEC26"
target_expiry = datetime.strptime(target_expiry_str, "%d%b%y").strftime("%Y%m%d")

currencies = ["BTC", "ETH"]
# currencies = ["BTC", "ETH", "HYPE", "SOL", "ZEC"]

https://alphasignal-dev.moretoncp.com


In [11]:
# 1. Get all dfs
orders_df = pd.read_parquet(f"{DATA_DIR}/orders.parquet")
fills_df = pd.read_parquet(f"{DATA_DIR}/fills.parquet")
positions_df = pd.read_parquet(f"{DATA_DIR}/positions.parquet")
realized_pnl_df = pd.read_parquet(f"{DATA_DIR}/realized_pnl.parquet") # will be overwritten
equity_df = pd.read_parquet(f"{DATA_DIR}/equity.parquet")

# fills_df = fills_df.iloc[0:0]

In [ ]:
# BACKTEST

# # 1. Get all dfs
# orders_df_backtest = pd.read_parquet(f"{DATA_DIR}/orders_backtest.parquet")
# fills_df_backtest = pd.read_parquet(f"{DATA_DIR}/fills_backtest.parquet")
# positions_df_backtest = pd.read_parquet(f"{DATA_DIR}/positions_backtest.parquet")
# realized_pnl_df_backtest = pd.read_parquet(f"{DATA_DIR}/realized_pnl_backtest.parquet") # will be overwritten
# equity_df_backtest = pd.read_parquet(f"{DATA_DIR}/equity_backtest.parquet")

# fills_df_backtest = portfolio.sync_fills_test(api=api, fills_df=fills_df)
# positions_df_backtest, realized_pnl_df_backtest = portfolio.reconstruct_positions_fifo(fills_df=fills_df_backtest)
# positions_df_backtest = portfolio.mark_positions_to_market(api=api, positions_df=positions_df_backtest)
# equity_df_backtest = portfolio.calculate_equity(api=api, positions_df=positions_df_backtest, 
#                                 realized_pnl_df=realized_pnl_df_backtest, equity_df=equity_df_backtest)

In [12]:
# 2. Get newly executed trades
fills_df = portfolio.sync_fills(api=api, fills_df=fills_df)
fills_df

,order_id,condition_id,token_id,outcome,side,price,shares,fee_rate,fee_rate_bps,timestamp,fill_id
0,0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,BUY,0.981,4.87,0.07,0.0,2026-08-14 08:25:11+00:00,4c3c68e2-7b89-463c-ae03-48b92808eeef
1,0xf7ff5b16da1bb114dc0b34d7e83324beb8b85502f029...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,BUY,0.969,4.91,0.07,0.0,2026-08-19 08:01:06+00:00,cd0ebe33-7e52-44c6-a022-466914d90934
2,0x387e06d4de5bb3110d135297db63ce667c4a3ac0de7d...,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,BUY,0.960,4.95,0.07,0.0,2026-08-19 08:00:45+00:00,ff80c017-a13d-43ee-85c0-35a2c289c1b6


In [13]:
# 3. Reconstruct portfolio
# 4. Calculate realized P&L
positions_df, realized_pnl_df = portfolio.reconstruct_positions_fifo(fills_df=fills_df) # new realized_df to overwrite old
realized_pnl_df

,condition_id,token_id,outcome,realized_shares,realized_pnl,realized_fees
0,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,0.0,0.0,0.0
1,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,0.0,0.0,0.0
2,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,0.0,0.0,0.0


In [14]:
# 5. Sync positions with api
portfolio.reconcile_positions(api=api, positions_df=positions_df)

POSITIONS SYNCED
POSITIONS SYNCED
POSITIONS SYNCED


In [15]:
# 6. Get latest order state
orders_df = portfolio.sync_orders(api=api, orders_df=orders_df)
orders_df

,order_id,condition_id,token_id,outcome,side,price,requested_size,order_type,status,created_at,cancelled_at
0,0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,NO,buy,0.981,4.877934,GTC,OPEN,2026-08-14 09:28:01.978131+00:00,NaT
1,0x387e06d4de5bb3110d135297db63ce667c4a3ac0de7d...,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,NO,buy,0.960,4.959176,GTC,OPEN,2026-08-19 08:00:40.748876+00:00,NaT
2,0xf7ff5b16da1bb114dc0b34d7e83324beb8b85502f029...,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,NO,buy,0.969,4.913116,GTC,OPEN,2026-08-19 08:01:02.498064+00:00,NaT


In [17]:
# 7. Mark positions to market
positions_df = portfolio.mark_positions_to_market(api=api, positions_df=positions_df)
positions_df

,condition_id,token_id,outcome,shares,cost_basis,avg_entry_price,realized_pnl,realized_shares,realized_fees,current_price,market_value,unrealized_pnl,unrealized_return
0,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,4251240413067872674686064123803499349976238079...,No,4.95,4.765306,0.962688,0.0,0.0,0.0,0.950,4.70250,-0.062806,-0.013180
1,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,4479131210875376621873271572183009281689857070...,No,4.91,4.768114,0.971103,0.0,0.0,0.0,0.968,4.75288,-0.015234,-0.003195
2,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,4.87,4.783824,0.982305,0.0,0.0,0.0,0.982,4.78234,-0.001484,-0.000310


In [18]:
# 8. Calculate equity
equity_df = portfolio.calculate_equity(api=api, positions_df=positions_df, 
                                realized_pnl_df=realized_pnl_df, equity_df=equity_df)
equity_df

Equity:  99.92
Return:  0.0000%
Sharpe: N/A
Sortino: N/A


,timestamp,cash,market_value,equity,realized_pnl,unrealized_pnl,daily_return,period_return,sharpe,sortino
0,2026-08-14 07:47:24.047885+00:00,100.00000,0.00000,100.00000,0.0,0.000000,NaN,NaN,NaN,NaN
1,2026-08-15 14:14:05.272655+00:00,95.21618,4.79695,100.01313,0.0,0.019480,NaN,0.000131,NaN,NaN
2,2026-08-17 02:58:57.343900+00:00,95.21618,4.77747,99.99365,0.0,0.000000,NaN,-0.000195,NaN,NaN
3,2026-08-19 07:56:33.538278+00:00,95.21618,4.78234,99.99852,0.0,0.004870,NaN,0.000049,NaN,NaN
4,2026-08-19 08:05:37.263592+00:00,85.68277,14.23772,99.92049,0.0,-0.049540,NaN,-0.000780,NaN,NaN
5,2026-08-19 11:33:46.319333+00:00,85.68277,14.23772,99.92049,0.0,-0.079524,NaN,0.000000,NaN,NaN


In [12]:
# 9. Get current markets
all_markets_df = api.get_all_markets(DATA_DIR=DATA_DIR, count_limit=10, liquidity_num_min=10000, volume_num_min=5000)
all_markets_df.head()

Fetched 100 markets | Total: 100
Fetched 100 markets | Total: 200
Fetched 100 markets | Total: 300
Fetched 100 markets | Total: 400
Fetched 100 markets | Total: 500
Fetched 100 markets | Total: 600
Fetched 100 markets | Total: 700
Fetched 100 markets | Total: 800
Fetched 100 markets | Total: 900
Fetched 100 markets | Total: 1,000


,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,version,negRiskMarketID,seriesColor,showGmpSeries,showGmpOutcome,umaResolutionStatus,oneHourPriceChange,eventStartTime,gameStartTime,groupItemRange
0,559651,Xi Jinping out before 2027?,0xa467b14d51f01b957109d9cbb1d6c124fab2a089d52e...,xi-jinping-out-before-2027,,2026-12-31T00:00:00Z,265073.20748,2025-07-03T20:37:00.228Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN
1,559652,Will Gavin Newsom win the 2028 Democratic pres...,0x0f49db97f71c68b1e42a6d16e3de93d85dbf7d4148e3...,will-gavin-newsom-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,301250.22354,2025-07-11T18:35:56.805Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
2,559653,Will Alexandria Ocasio-Cortez win the 2028 Dem...,0xe6bcc2f1dd025ce5e1833190f7c60a71171c94f805df...,will-alexandria-ocasio-cortez-win-the-2028-dem...,,2028-11-07T00:00:00Z,360252.86913,2025-07-11T18:35:59.075Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
3,559654,Will Pete Buttigieg win the 2028 Democratic pr...,0x4c325469d9b516ef4e6b8f73a81a12607dec075e3c2f...,will-pete-buttigieg-win-the-2028-democratic-pr...,,2028-11-07T00:00:00Z,528145.823,2025-07-11T18:35:58.818Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN
4,559655,Will Josh Shapiro win the 2028 Democratic pres...,0xd65891729ce093cc12236856837eba1a0872fc7998fd...,will-josh-shapiro-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,483420.49813,2025-07-11T18:36:01.098Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,v1,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,,False,False,NaN,NaN,NaN,NaN,NaN


In [13]:
# 10. Initialize variance surface
deribit = Deribit(currencies=currencies, target_expiry=target_expiry)
okx = OKX(currencies=currencies, target_expiry=target_expiry)
bybit = Bybit(currencies=currencies, target_expiry=target_expiry)

s.initialize(currencies=currencies, exchanges=[deribit, okx, bybit])

spot: 64383.0 volume24h: 0.6203
spot: 1919.6 volume24h: 25.5051
spot: 64389.2 volume24h: 3978.75931279
spot: 1918.35 volume24h: 52864.473492
spot: 64382.6 volume24h: 5758.963178
spot: 1918.28 volume24h: 54187.0553


In [14]:
# 11. Scan markets
markets_df, opportunities_df = scanner.scan_market(markets_df=all_markets_df, s=s, crypto_tag_ids=crypto_tag_ids)
opportunities_df.head()

question: Will Bitcoin hit $150k by December 31, 2026?
event type: touch
direction: up
currency: BTC
required strike: 150000.0
iv: 0.5128221544606524
buy_yes_ev: -0.009752223374581929
sell_yes_ev: 0.00702413337458193
buy_no_ev: 0.0070241333745819
sell_no_ev: -0.009752223374581974
buy_yes_kelly: -0.004944835856648765
sell_yes_kelly: 0.314417326222459
buy_no_kelly: 0.3144173262224585
sell_no_kelly: -0.004944835856648787

question: Will Bitcoin reach $200,000 by December 31, 2026?
event type: touch
direction: up
currency: BTC
required strike: 200000.0
iv: 0.5866136351633732
buy_yes_ev: -0.018454256582258875
sell_yes_ev: 0.015047166582258874
buy_no_ev: 0.01504716658225893
sell_no_ev: -0.01845425658225888
buy_yes_kelly: -0.009408115214099946
sell_yes_kelly: 0.47526683384445056
buy_no_kelly: 0.4752668338444506
sell_no_kelly: -0.00940811521409995

question: Will Bitcoin reach $190,000 by December 31, 2026?
event type: touch
direction: up
currency: BTC
required strike: 190000.0
iv: 0.577103781

,id,question,endDate,yes_ask,no_ask,model_prob,conditionId,slug,resolutionSource,liquidity,...,buy_yes_ev,sell_yes_ev,buy_no_ev,sell_no_ev,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
743,701552,"Will Ethereum dip to $1,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.402,0.599,0.522990,0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...,will-ethereum-dip-to-1500-by-december-31-2026-...,,70257.75215,...,0.104162,-0.138804,-0.138804,0.104162,0.089614,-0.180647,-0.180647,0.089614,0.104162,buy_yes_ev
729,701502,"Will Bitcoin dip to $45,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.22,0.79,0.288855,0x024b68f77bfc019341ee3db8f57c103334e4b9430bba...,will-bitcoin-dip-to-45000-by-december-31-2026-...,,164897.6908,...,0.056843,-0.090468,-0.090468,0.056843,0.037008,-0.228010,-0.228010,0.037008,0.056843,buy_yes_ev
728,701501,"Will Bitcoin dip to $55,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.54,0.47,0.592828,0x752fa61c93f16c4b15e85b0bd438d9c684176b0ea7b3...,will-bitcoin-dip-to-55000-by-december-31-2026-...,,99844.3352,...,0.035440,-0.080265,-0.080265,0.035440,0.040036,-0.078298,-0.078298,0.040036,0.035440,buy_yes_ev
730,701503,"Will Bitcoin dip to $35,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.1,0.91,0.141137,0x2745c38ff0617cb345c1d2df19b4f74ea777508e0741...,will-bitcoin-dip-to-35000-by-december-31-2026-...,,109998.0282,...,0.034837,-0.056870,-0.056870,0.034837,0.019490,-0.337439,-0.337439,0.019490,0.034837,sell_no_ev
731,701504,"Will Bitcoin dip to $25,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.032,0.971,0.067107,0xe326d1abf5fb59b82ecfdff3348e75f90561eace327b...,will-bitcoin-dip-to-25000-by-december-31-2026-...,,112247.15921,...,0.032939,-0.040078,-0.040078,0.032939,0.017052,-0.741394,-0.741394,0.017052,0.032939,buy_yes_ev


In [14]:
opportunities_df

,id,question,endDate,yes_ask,no_ask,model_prob,conditionId,slug,resolutionSource,liquidity,...,buy_yes_ev,sell_yes_ev,buy_no_ev,sell_no_ev,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
585,701552,"Will Ethereum dip to $1,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.402,0.599,0.525600,0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...,will-ethereum-dip-to-1500-by-december-31-2026-...,,80206.24887,...,0.106772,-0.141414,-0.141414,0.106772,0.091859,-0.184044,-0.184044,0.091859,0.106772,buy_yes_ev
571,701502,"Will Bitcoin dip to $45,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.22,0.79,0.290404,0x024b68f77bfc019341ee3db8f57c103334e4b9430bba...,will-bitcoin-dip-to-45000-by-december-31-2026-...,,176419.5778,...,0.058392,-0.092017,-0.092017,0.058392,0.038016,-0.231913,-0.231913,0.038016,0.058392,buy_yes_ev
570,701501,"Will Bitcoin dip to $55,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.54,0.47,0.596012,0x752fa61c93f16c4b15e85b0bd438d9c684176b0ea7b3...,will-bitcoin-dip-to-55000-by-december-31-2026-...,,109562.3945,...,0.038624,-0.083449,-0.083449,0.038624,0.043631,-0.081403,-0.081403,0.043631,0.038624,buy_yes_ev
572,701503,"Will Bitcoin dip to $35,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.1,0.91,0.142061,0x2745c38ff0617cb345c1d2df19b4f74ea777508e0741...,will-bitcoin-dip-to-35000-by-december-31-2026-...,,121810.5132,...,0.035761,-0.057794,-0.057794,0.035761,0.020007,-0.342919,-0.342919,0.020007,0.035761,sell_no_ev
573,701504,"Will Bitcoin dip to $25,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.032,0.971,0.067666,0xe326d1abf5fb59b82ecfdff3348e75f90561eace327b...,will-bitcoin-dip-to-25000-by-december-31-2026-...,,119892.99113,...,0.033497,-0.040637,-0.040637,0.033497,0.017341,-0.751729,-0.751729,0.017341,0.033497,buy_yes_ev
586,701553,"Will Ethereum dip to $1,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.12,0.9,0.154152,0xacb33346b59a2a3770e2391b7d1b0e77d8dcdcf840a6...,will-ethereum-dip-to-1000-by-december-31-2026-...,,107512.879,...,0.026760,-0.060452,-0.060452,0.026760,0.015333,-0.322581,-0.322581,0.015333,0.026760,sell_no_ev
582,701547,"Will Ethereum reach $4,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.05,0.96,0.011920,0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...,will-ethereum-reach-4500-by-december-31-2026,,44865.9755,...,-0.041405,0.025392,0.025392,-0.041405,-0.021868,0.340260,0.340260,-0.021868,0.025392,buy_no_ev
580,701545,"Will Ethereum reach $5,500 by December 31, 2026?",2027-01-01T05:00:00Z,0.032,0.969,0.004079,0xb75b8c777c5fd934d7fa43d4b3a3f630981b68a362a0...,will-ethereum-reach-5500-by-december-31-2026,,32265.63124,...,-0.030089,0.024818,0.024818,-0.030089,-0.015577,0.429418,0.429418,-0.015577,0.024818,sell_yes_ev
583,701548,"Will Ethereum reach $4,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.06,0.95,0.023267,0x9775cd557a3cdba4f0478070afa399c0805761dd9aa0...,will-ethereum-reach-4000-by-december-31-2026,,41181.0932,...,-0.040681,0.023408,0.023408,-0.040681,-0.021730,0.250750,0.250750,-0.021730,0.023408,buy_no_ev
567,701494,"Will Bitcoin reach $120,000 by December 31, 2026?",2027-01-01T05:00:00Z,0.05,0.96,0.014509,0xf9b6b5c3c6e07afe4aad6ed9ce4fa7545f9c2baa9f1d...,will-bitcoin-reach-120000-by-december-31-2026-...,,52908.6704,...,-0.038816,0.022803,0.022803,-0.038816,-0.020501,0.305571,0.305571,-0.020501,0.022803,buy_no_ev


In [15]:
# 12. manage cancel orders
orders_df = portfolio.manage_open_orders(api=api, orders_df=orders_df, markets_df=markets_df, 
                ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)
orders_df

STATUS: 502
RESPONSE: {"detail":"Polymarket CLOB error: Request exception!"}
REQUEST URL: https://alphasignal-dev.moretoncp.com/v1/orders
REQUEST BODY: None
REQUEST PARAMS: {}


HTTPError: 502 Server Error: Bad Gateway for url: https://alphasignal-dev.moretoncp.com/v1/orders

In [16]:
# 13. Risk management
orders_df = portfolio.run_risk_management(api=api, positions_df=positions_df, 
            markets_df=markets_df, orders_df=orders_df, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

pos: condition_id         0xa3143f9c21ecc7a9b036b4e435f48264ff72019a9e3e...
token_id             4251240413067872674686064123803499349976238079...
outcome                                                             No
shares                                                            4.95
cost_basis                                                       4.752
avg_entry_price                                                   0.96
realized_pnl                                                       0.0
realized_shares                                                    0.0
realized_fees                                                      0.0
current_price                                                     0.95
market_value                                                    4.7025
unrealized_pnl                                                 -0.0495
unrealized_return                                            -0.010417
Name: 0, dtype: object
market: id                                       

In [16]:
# 14. New opportunities
orders_df = portfolio.run_new_opportunities(api=api, opportunities_df=opportunities_df, orders_df=orders_df, 
            ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, MAX_POSITION=MAX_POSITION, FRACTION=FRACTION)

current balance size: 0.0
dollars: 2.1866264638395267
BUY SIGNAL
Market                    : Will Ethereum dip to $1,500 by December 31, 2026?
Outcome                   : YES
Direction                 : down
Event Type                : touch
Best Action               : buy_yes_ev
Kelly Size x Fraction     : 0.022964862314782286
Recommended Size          : 5.439369
Current Ask               : 0.402
P Yes                     : 0.5256000511309448
P No                      : 0.47439994886905523
Buy Yes EV                : 0.10677233113094478
Sell No EV                : 0.10677233113094478
Buy No EV                 : -0.14141398113094472
Sell Yes EV               : -0.14141398113094472
Unrealized Pnl            : 0.580774108011396
Cash                      : 95.21618
Current Shares            : 0.0
Fee Rate                  : 0.07
ENTRY EV THRESHOLD        : 0.02
MAX POSITION THRESHOLD    : 0.05
                         VARIABLES IN REQUEST                         
tokenId                  

In [45]:
# 15. Safe dfs
portfolio.save_snapshots(DATA_DIR, markets_df, "markets")
portfolio.save_snapshots(DATA_DIR, opportunities_df, "opportunities")
portfolio.save(DATA_DIR, orders_df, "orders")
portfolio.save(DATA_DIR, fills_df, "fills")
portfolio.save(DATA_DIR, positions_df, "positions")
portfolio.save(DATA_DIR, realized_pnl_df, "realized_pnl")
portfolio.save(DATA_DIR, equity_df, "equity")

save_snapshots df saved at: data/20260819/markets_20260819_165010.parquet
save_snapshots df saved at: data/20260819/opportunities_20260819_165010.parquet
save df saved at:  data/orders.parquet
save df saved at:  data/fills.parquet
save df saved at:  data/positions.parquet
save df saved at:  data/realized_pnl.parquet
save df saved at:  data/equity.parquet


In [ ]:
# BACKTEST

# # 13. Risk management
# orders_df_backtest = portfolio.run_risk_management_test(api=api, positions_df=positions_df_backtest, 
#             markets_df=markets_df_backtest, orders_df=orders_df_backtest, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

# # 14. New opportunities
# orders_df_backtest = portfolio.run_new_opportunities_test(api=api, opportunities_df=opportunities_df_backtest, orders_df=orders_df_backtest, 
#             ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, MAX_POSITION=MAX_POSITION, FRACTION=FRACTION)

# # 15. Safe dfs
# portfolio.save_snapshots(DATA_DIR, markets_df_backtest, "markets_backtest")
# portfolio.save_snapshots(DATA_DIR, opportunities_df_backtest, "opportunities_backtest")
# portfolio.save(DATA_DIR, orders_df_backtest, "orders_backtest")
# portfolio.save(DATA_DIR, fills_df_backtest, "fills_backtest")
# portfolio.save(DATA_DIR, positions_df_backtest, "positions_backtest")
# portfolio.save(DATA_DIR, realized_pnl_df_backtest, "realized_pnl_backtest")
# portfolio.save(DATA_DIR, equity_df_backtest, "equity_backtest")

In [ ]:
ENTRY_EV_THRESHOLD = 0.02   # require 1% edge, default = 0
EXIT_EV_THRESHOLD = 0.02
FRACTION = 0.25
MAX_POSITION = 0.05

load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
JWT_TOKEN = os.getenv("JWT")
DATA_DIR = os.getenv("DATA_DIR")
print(BASE_URL)

with open(f"{DATA_DIR}/crypto_tag_ids.json", "r") as f:
    crypto_tag_ids = set(json.load(f))

s = OptionSurface()
portfolio = Portfolio()
api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD, token=JWT_TOKEN)
scanner = MarketScanner(api=api)

target_expiry_str = "25DEC26"
target_expiry = datetime.strptime(target_expiry_str, "%d%b%y").strftime("%Y%m%d")

currencies = ["BTC", "ETH"]

for i in range(1):
    # 1. Get all dfs
    orders_df = pd.read_parquet(f"{DATA_DIR}/orders.parquet")
    fills_df = pd.read_parquet(f"{DATA_DIR}/fills.parquet")
    positions_df = pd.read_parquet(f"{DATA_DIR}/positions.parquet")
    equity_df = pd.read_parquet(f"{DATA_DIR}/equity.parquet")

    # 2. Get newly executed trades
    fills_df = portfolio.sync_fills(api=api, fills_df=fills_df)

    # 3. Reconstruct portfolio
    # 4. Calculate realized P&L
    positions_df, realized_pnl_df = portfolio.reconstruct_positions_fifo(fills_df=fills_df) # new realized_df to overwrite old

    # 5. Sync positions with api
    portfolio.reconcile_positions(api=api, positions_df=positions_df)

    # 6. Get latest order state
    orders_df = portfolio.sync_orders(api=api, orders_df=orders_df)

    # 7. Mark positions to market
    positions_df = portfolio.mark_positions_to_market(api=api, positions_df=positions_df)

    # 8. Calculate equity
    equity_df = portfolio.calculate_equity(api=api, positions_df=positions_df, 
                                    realized_pnl_df=realized_pnl_df, equity_df=equity_df)

    # 9. Get current markets
    all_markets_df = api.get_all_markets(DATA_DIR=DATA_DIR, count_limit=10, liquidity_num_min=10000, volume_num_min=5000)

    # 10. Initialize variance surface
    deribit = Deribit(currencies=currencies, target_expiry=target_expiry)
    okx = OKX(currencies=currencies, target_expiry=target_expiry)
    bybit = Bybit(currencies=currencies, target_expiry=target_expiry)

    s.initialize(currencies=currencies, exchanges=[deribit, okx, bybit])

    # 11. Scan markets
    markets_df, opportunities_df = scanner.scan_market(markets_df=all_markets_df, s=s, crypto_tag_ids=crypto_tag_ids)

    # 12. manage cancel orders
    orders_df = portfolio.manage_open_orders(api=api, orders_df=orders_df, markets_df=markets_df, 
                ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

    # 13. Risk management
    orders_df = portfolio.run_risk_management(api=api, positions_df=positions_df, 
            markets_df=markets_df, orders_df=orders_df, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

    # 14. New opportunities
    orders_df = portfolio.run_new_opportunities(api=api, opportunities_df=opportunities_df, orders_df=orders_df, 
            ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, MAX_POSITION=MAX_POSITION, FRACTION=FRACTION)

    # 15. Safe dfs
    portfolio.save_snapshots(DATA_DIR, markets_df, "markets")
    portfolio.save_snapshots(DATA_DIR, opportunities_df, "opportunities")
    portfolio.save(DATA_DIR, orders_df, "orders")
    portfolio.save(DATA_DIR, fills_df, "fills")
    portfolio.save(DATA_DIR, positions_df, "positions")
    portfolio.save(DATA_DIR, realized_pnl_df, "realized_pnl")
    portfolio.save(DATA_DIR, equity_df, "equity")

    # time.sleep(300)

In [ ]:
# Polymarket    Model (exchanges: deribit, bybit, okx)    Edge

# 80k Dec-26        22%          20%       +2%
# 90k Dec-26        14%          12%       +2%
# 100k Dec-26        9.5%         5.9%     +3.6%
# 110k Dec-26        7%           4%       +3%
# 120k Dec-26        5%           2%       +3%

In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4